# FashionSigLIP2: HSC fine-tuning and evaluation

Cleaned/deduplicated version of the original exploratory notebook. Structure:

1. Setup
2. Frozen baseline benchmark (no fine-tuning) — compares candidate checkpoints on caption retrieval
3. Zero-shot hierarchical softmax classification (HSC) climbing demo over a manually defined taxonomy
4. Fine-tuning: hierarchical multi-positive SigLIP2 with identity-balanced (P products × K images) batches
5. Evaluation: fine-tuned vs. base SigLIP2 across label granularities (exact product, brand, attribute, generic, full caption)

**Repointed at the current dataset (2026-07-27):** `DATASET_ROOT` now points at `/content/drive/MyDrive/apparel_dataset` (1115 records, 6 brands), read from Google Drive via Colab — not from local disk. The fine-tuning section's label generation (§4) now reads each product's `structured_caption` (taxonomy_path + attributes + positive_texts) directly instead of regex-parsing the legacy flat `caption` string — this removes the old shoe-only category/color/material detection heuristics entirely, since the LLM captioner already produces that structure for all 6 brands. The frozen-baseline benchmark (§2) still scores against the flat `caption` field for now; upgrading it to use `structured_caption` positive texts is a Phase 1 task, not part of this repointing pass.


## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -U transformers


## 2. Frozen baseline benchmark

Per the project spec's evaluation-before-fine-tuning rule: compare the prior fine-tuned checkpoint (`srpone/zooclaw-fashionsiglip2`) against the unmodified base SigLIP2 checkpoint on image-to-caption retrieval, before drawing any conclusions about fine-tuning gains.

In [ ]:
# Optional for a fresh Colab runtime:
# !pip install -q -U transformers accelerate sentencepiece pillow pandas tqdm

from pathlib import Path
import gc
import json

import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image, ImageOps
from tqdm.auto import tqdm
from transformers import AutoModel, AutoProcessor


# ============================================================
# Configuration
# ============================================================

DATASET_ROOT = Path("/content/drive/MyDrive/apparel_dataset")
METADATA_PATH = DATASET_ROOT / "metadata.json"

OUTPUT_DIR = DATASET_ROOT / "siglip2_caption_benchmark"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_IDS = {
    "finetuned": "srpone/zooclaw-fashionsiglip2",

    # Fair baseline because ZooClaw was fine-tuned from this model.
    "base": "google/siglip2-base-patch16-384",

    # Use this instead only if you specifically want the 224px model:
    # "base": "google/siglip2-base-patch16-224",
}

IMAGE_BATCH_SIZE = 32
TEXT_BATCH_SIZE = 128
SIMILARITY_BATCH_SIZE = 512

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

print("Device:", DEVICE)
print("Dtype:", DTYPE)


In [ ]:


# ============================================================
# Load metadata
# ============================================================

with METADATA_PATH.open("r", encoding="utf-8") as f:
    metadata = json.load(f)


def resolve_image_path(raw_path):
    """
    Supports paths such as:

    apparel_dataset/adidas/.../image_0.jpg
    adidas/.../image_0.jpg
    /absolute/path/image_0.jpg
    """

    raw_path = Path(raw_path)

    candidates = [
        raw_path,
        DATASET_ROOT / raw_path,
        DATASET_ROOT.parent / raw_path,
    ]

    # Prevent:
    # /apparel_dataset/apparel_dataset/adidas/...
    if raw_path.parts and raw_path.parts[0] == DATASET_ROOT.name:
        candidates.append(
            DATASET_ROOT.joinpath(*raw_path.parts[1:])
        )

    # Real files always live at DATASET_ROOT/<brand>/<slug>/<product_code>/image_N.jpg.
    # Some products still carry a stale prefix (e.g. "shoe_dataset/...") baked into
    # metadata.json from before the apparel_dataset rename -- reconstruct from the
    # last 4 path components regardless of what prefix is actually present.
    if len(raw_path.parts) >= 4:
        candidates.append(
            DATASET_ROOT.joinpath(*raw_path.parts[-4:])
        )

    for candidate in candidates:
        if candidate.is_file():
            return candidate.resolve()

    return None


In [ ]:
# ============================================================
# Build image records and candidate captions
# ============================================================

# Exact duplicate captions are collapsed into one candidate label.
captions = []
caption_to_index = {}

image_records = []
missing_paths = []
skipped_products = []

for product in metadata:
    caption = str(product.get("caption", "")).strip()
    image_paths = product.get("images") or []

    if not caption or not image_paths:
        skipped_products.append(product.get("product_code"))
        continue

    if caption not in caption_to_index:
        caption_to_index[caption] = len(captions)
        captions.append(caption)

    caption_index = caption_to_index[caption]

    for raw_path in image_paths:
        resolved_path = resolve_image_path(raw_path)

        if resolved_path is None:
            missing_paths.append(raw_path)
            continue

        image_records.append({
            "image_path": str(resolved_path),
            "caption_index": caption_index,
            "caption": caption,
            "product_code": product.get("product_code", ""),
            "brand": product.get("brand", ""),
            "name": product.get("name", ""),
        })


if not image_records:
    raise RuntimeError(
        "No valid images were found. Check DATASET_ROOT and metadata paths."
    )

if not captions:
    raise RuntimeError("No valid captions were found.")


print(f"Products in metadata:      {len(metadata):,}")
print(f"Candidate captions:        {len(captions):,}")
print(f"Evaluation images:         {len(image_records):,}")
print(f"Missing image paths:       {len(missing_paths):,}")
print(f"Skipped products:          {len(skipped_products):,}")


if missing_paths:
    missing_report = OUTPUT_DIR / "missing_image_paths.txt"
    missing_report.write_text(
        "\n".join(map(str, missing_paths)),
        encoding="utf-8",
    )
    print("Missing-path report:", missing_report)


In [ ]:

# ============================================================
# Embedding functions
# ============================================================

def extract_embeddings(output):
    """Handle different Transformers SigLIP/SigLIP2 return formats."""

    if torch.is_tensor(output):
        return output

    if getattr(output, "text_embeds", None) is not None:
        return output.text_embeds

    if getattr(output, "image_embeds", None) is not None:
        return output.image_embeds

    if getattr(output, "pooler_output", None) is not None:
        return output.pooler_output

    if isinstance(output, (tuple, list)) and torch.is_tensor(output[0]):
        return output[0]

    raise TypeError(
        f"Could not extract embeddings from output type: {type(output)}"
    )


def move_to_device(inputs):
    return {
        key: value.to(DEVICE, non_blocking=True)
        if isinstance(value, torch.Tensor)
        else value
        for key, value in inputs.items()
    }


@torch.inference_mode()
def encode_texts(model, processor, texts):
    embeddings = []

    for start in tqdm(
        range(0, len(texts), TEXT_BATCH_SIZE),
        desc="Encoding captions",
    ):
        batch_texts = texts[start:start + TEXT_BATCH_SIZE]

        inputs = processor(
            text=batch_texts,
            padding="max_length",
            truncation=True,
            max_length=64,
            return_tensors="pt",
        )

        inputs = move_to_device(inputs)

        outputs = model.get_text_features(**inputs)
        batch_embeddings = extract_embeddings(outputs)

        batch_embeddings = F.normalize(
            batch_embeddings.float(),
            p=2,
            dim=-1,
        )

        embeddings.append(batch_embeddings.cpu())

    return torch.cat(embeddings, dim=0)


@torch.inference_mode()
def encode_images(model, processor, records):
    embeddings = []
    valid_records = []
    failed_images = []

    for start in tqdm(
        range(0, len(records), IMAGE_BATCH_SIZE),
        desc="Encoding images",
    ):
        batch_records = records[start:start + IMAGE_BATCH_SIZE]

        images = []
        kept_records = []

        for record in batch_records:
            try:
                with Image.open(record["image_path"]) as image:
                    image = ImageOps.exif_transpose(image)
                    image = image.convert("RGB")
                    images.append(image)

                kept_records.append(record)

            except Exception as error:
                failed_images.append({
                    "image_path": record["image_path"],
                    "error": repr(error),
                })

        if not images:
            continue

        inputs = processor(
            images=images,
            return_tensors="pt",
        )

        inputs = move_to_device(inputs)

        outputs = model.get_image_features(**inputs)
        batch_embeddings = extract_embeddings(outputs)

        batch_embeddings = F.normalize(
            batch_embeddings.float(),
            p=2,
            dim=-1,
        )

        embeddings.append(batch_embeddings.cpu())
        valid_records.extend(kept_records)

    if not embeddings:
        raise RuntimeError("Every image failed to load or encode.")

    return (
        torch.cat(embeddings, dim=0),
        valid_records,
        failed_images,
    )


In [ ]:

# ============================================================
# Image-to-caption evaluation
# ============================================================

def evaluate_image_to_caption(
    image_embeddings,
    text_embeddings,
    records,
    candidate_captions,
):
    if len(image_embeddings) != len(records):
        raise ValueError(
            "Image embedding count does not match record count."
        )

    target_indices = torch.tensor(
        [record["caption_index"] for record in records],
        dtype=torch.long,
    )

    maximum_top_k = min(10, len(candidate_captions))

    all_ranks = []
    prediction_rows = []

    for start in tqdm(
        range(0, len(records), SIMILARITY_BATCH_SIZE),
        desc="Scoring images",
    ):
        end = min(
            start + SIMILARITY_BATCH_SIZE,
            len(records),
        )

        # Both embedding matrices are normalized, so this is cosine similarity.
        scores = image_embeddings[start:end] @ text_embeddings.T
        targets = target_indices[start:end]

        target_scores = scores[
            torch.arange(end - start),
            targets,
        ]

        # Rank 1 means no other caption has a higher similarity.
        ranks = (
            scores > target_scores[:, None]
        ).sum(dim=1) + 1

        all_ranks.append(ranks)

        top_scores, top_indices = scores.topk(
            k=maximum_top_k,
            dim=1,
        )

        for local_index, global_index in enumerate(
            range(start, end)
        ):
            record = records[global_index]

            predicted_index = int(
                top_indices[local_index, 0]
            )

            target_index = int(targets[local_index])

            row = {
                **record,
                "rank": int(ranks[local_index]),
                "target_score": float(
                    target_scores[local_index]
                ),
                "predicted_caption_index": predicted_index,
                "predicted_caption":
                    candidate_captions[predicted_index],
                "predicted_score": float(
                    top_scores[local_index, 0]
                ),
                "correct_top1":
                    predicted_index == target_index,
            }

            for k in (1, 5, 10):
                effective_k = min(k, maximum_top_k)

                row[f"correct_top{k}"] = bool(
                    (
                        top_indices[
                            local_index,
                            :effective_k
                        ] == target_index
                    ).any()
                )

            prediction_rows.append(row)

    ranks = torch.cat(all_ranks).float()

    metrics = {
        "num_images": int(len(records)),
        "num_candidate_captions": int(
            len(candidate_captions)
        ),
        "recall_at_1": float(
            (ranks <= 1).float().mean()
        ),
        "recall_at_5": float(
            (
                ranks <= min(5, len(candidate_captions))
            ).float().mean()
        ),
        "recall_at_10": float(
            (
                ranks <= min(10, len(candidate_captions))
            ).float().mean()
        ),
        "mrr": float(
            (1.0 / ranks).mean()
        ),
        "median_rank": float(
            ranks.median()
        ),
        "mean_rank": float(
            ranks.mean()
        ),
    }

    return metrics, pd.DataFrame(prediction_rows)

In [ ]:

# ============================================================
# Evaluate both models
# ============================================================

all_metrics = []

for model_name, model_id in MODEL_IDS.items():
    print("\n" + "=" * 80)
    print(f"Evaluating {model_name}: {model_id}")
    print("=" * 80)

    processor = AutoProcessor.from_pretrained(model_id)

    model = AutoModel.from_pretrained(
        model_id,
        torch_dtype=DTYPE,
    )

    model = model.to(DEVICE)
    model.eval()

    text_embeddings = encode_texts(
        model=model,
        processor=processor,
        texts=captions,
    )

    (
        image_embeddings,
        valid_records,
        failed_images,
    ) = encode_images(
        model=model,
        processor=processor,
        records=image_records,
    )

    metrics, predictions = evaluate_image_to_caption(
        image_embeddings=image_embeddings,
        text_embeddings=text_embeddings,
        records=valid_records,
        candidate_captions=captions,
    )

    metrics["model_name"] = model_name
    metrics["model_id"] = model_id
    metrics["failed_images"] = len(failed_images)

    all_metrics.append(metrics)

    predictions.insert(
        0,
        "model_id",
        model_id,
    )

    predictions.to_csv(
        OUTPUT_DIR /
        f"{model_name}_per_image_predictions.csv",
        index=False,
    )

    if failed_images:
        pd.DataFrame(failed_images).to_csv(
            OUTPUT_DIR /
            f"{model_name}_failed_images.csv",
            index=False,
        )

    with (
        OUTPUT_DIR / f"{model_name}_metrics.json"
    ).open("w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2)

    print("\nMetrics:")
    print(pd.Series(metrics).to_string())

    # Release GPU memory before loading the next checkpoint.
    del model
    del processor
    del text_embeddings
    del image_embeddings

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ============================================================
# Final comparison
# ============================================================

summary = pd.DataFrame(all_metrics)[[
    "model_name",
    "model_id",
    "num_images",
    "num_candidate_captions",
    "recall_at_1",
    "recall_at_5",
    "recall_at_10",
    "mrr",
    "median_rank",
    "mean_rank",
    "failed_images",
]]

percentage_columns = [
    "recall_at_1",
    "recall_at_5",
    "recall_at_10",
    "mrr",
]

for column in percentage_columns:
    summary[column] *= 100.0

summary = summary.rename(columns={
    "recall_at_1": "R@1 (%)",
    "recall_at_5": "R@5 (%)",
    "recall_at_10": "R@10 (%)",
    "mrr": "MRR (%)",
})

summary.to_csv(
    OUTPUT_DIR / "model_comparison.csv",
    index=False,
)

print("\nFinal comparison:")
display(summary)

print("\nResults saved to:")
print(OUTPUT_DIR)

## 3. Zero-shot HSC climbing demo

Manually defined category/color/brand hierarchy (`fashion_item -> footwear/clothing/accessory -> ... -> leaf`). For a single test image, scores every leaf via cosine similarity, then climbs from the most probable leaf toward the root until cumulative descendant probability clears a confidence threshold — this is the specificity-backoff behavior the project spec requires at query time (§2.5), demonstrated zero-shot across several candidate checkpoints (finetuned SigLIP2, base SigLIP2, FashionCLIP, locally finetuned FashionCLIP).

Set `IMAGE_PATH` to a real image before running.

In [ ]:
import gc
from functools import lru_cache

import torch
import torch.nn.functional as F
from PIL import Image, ImageOps
from transformers import (
    AutoModel,
    AutoModelForZeroShotImageClassification,
    AutoProcessor,
)


IMAGE_PATH = "/content/Screenshot 2026-07-18 at 3.27.01 PM.png"

LOCAL_FASHIONCLIP_PATH = (
    "/content/drive/MyDrive/fashion_models/1_both_projection_heads"
)

MODEL_CONFIGS = {
    "finetuned_siglip2": {
        "model_id": "srpone/zooclaw-fashionsiglip2",
        "processor_id": "srpone/zooclaw-fashionsiglip2",
        "loader": AutoModel,
        "local": False,
        "temperature": 0.05,
    },
    "base_siglip2": {
        "model_id": "google/siglip2-base-patch16-384",
        "processor_id": "google/siglip2-base-patch16-384",
        "loader": AutoModel,
        "local": False,
        "temperature": 0.05,
    },
    "fashionclip": {
        "model_id": "patrickjohncyh/fashion-clip",
        "processor_id": "patrickjohncyh/fashion-clip",
        "loader": AutoModelForZeroShotImageClassification,
        "local": False,
        "temperature": 0.05,
    },
    "finetuned_fashionclip": {
        "model_id": LOCAL_FASHIONCLIP_PATH,
        # Reuse the original preprocessing/tokenization settings.
        "processor_id": "patrickjohncyh/fashion-clip",
        "loader": AutoModelForZeroShotImageClassification,
        "local": True,
        "temperature": 0.05,
    },
}



DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Higher thresholds produce broader predictions.
HSC_THRESHOLDS = [0.30, 0.50, 0.70, 0.85, 0.95]

TOP_K = 10


# ============================================================
# Hierarchy
#
# Every child must have an "is-a" relationship with its parent.
# Only leaf nodes are scored directly by the selected vision-language model.
# ============================================================

HIERARCHY = {
    "fashion_item": {
        "footwear": {
            "shoe": {
                "sneaker": {
                    "yellow_sneaker": {
                        "yellow_adidas_sneaker": {
                            "yellow_adidas_samba": {},
                            "yellow_adidas_gazelle": {},
                            "yellow_adidas_campus": {},

                        },
                        "yellow_nike_sneaker": {
                            "yellow_nike_air_force_1": {},
                            "yellow_nike_dunk": {},

                        },

                    },
                    "black_sneaker": {
                        "black_adidas_samba": {},
                        "black_nike_air_force_1": {},

                    },
                    "white_sneaker": {
                        "white_adidas_samba": {},
                        "white_adidas_gazelle": {},
                        "white_nike_air_force_1": {},

                    },
                    "red_sneaker": {

                    },
                    "blue_sneaker": {

                    },
                },
                "dress_shoe": {
                    "black_dress_shoe": {},
                    "brown_dress_shoe": {},
                },
            },
            "boot": {
                "black_boot": {},
                "brown_boot": {},
                "yellow_boot": {},
            },
        },
        "clothing": {
            "top": {
                "yellow_top": {
                    "yellow_tshirt": {},
                    "yellow_hoodie": {},
                    "yellow_sweater": {},
                },
                "black_top": {
                    "black_tshirt": {},
                    "black_hoodie": {},
                },
                "white_top": {
                    "white_tshirt": {},
                    "white_hoodie": {},
                },
            },
            "bottom": {
                "pants": {
                    "blue_jeans": {},
                    "black_pants": {},
                    "yellow_pants": {},
                },
                "shorts": {
                    "blue_denim_shorts": {},
                    "black_shorts": {},
                    "yellow_shorts": {},
                },
            },
        },
        "accessory": {
            "bag": {
                "backpack": {
                    "red_backpack": {},
                    "black_backpack": {},
                    "yellow_backpack": {},
                },
                "handbag": {
                    "red_handbag": {},
                    "black_handbag": {},
                    "yellow_handbag": {},
                },
            },
        },
    }
}


# Human-readable output names for every node.
NODE_LABELS = {
    "fashion_item": "a fashion item",

    "footwear": "footwear",
    "shoe": "a shoe",
    "sneaker": "a sneaker",

    "yellow_sneaker": "a yellow sneaker",
    "yellow_adidas_sneaker": "a yellow Adidas sneaker",
    "yellow_adidas_samba": "a yellow Adidas Samba sneaker",
    "yellow_adidas_gazelle": "a yellow Adidas Gazelle sneaker",
    "yellow_adidas_campus": "a yellow Adidas Campus sneaker",

    "yellow_nike_sneaker": "a yellow Nike sneaker",
    "yellow_nike_air_force_1": "a yellow Nike Air Force 1 sneaker",
    "yellow_nike_dunk": "a yellow Nike Dunk sneaker",

    "black_sneaker": "a black sneaker",
    "black_adidas_samba": "a black Adidas Samba sneaker",
    "black_nike_air_force_1": "a black Nike Air Force 1 sneaker",

    "white_sneaker": "a white sneaker",
    "white_adidas_samba": "a white Adidas Samba sneaker",
    "white_adidas_gazelle": "a white Adidas Gazelle sneaker",
    "white_nike_air_force_1": "a white Nike Air Force 1 sneaker",

    "red_sneaker": "a red sneaker",

    "blue_sneaker": "a blue sneaker",

    "dress_shoe": "a dress shoe",
    "black_dress_shoe": "a black dress shoe",
    "brown_dress_shoe": "a brown dress shoe",

    "boot": "a boot",
    "black_boot": "a black boot",
    "brown_boot": "a brown boot",
    "yellow_boot": "a yellow boot",

    "clothing": "clothing",
    "top": "a top",

    "yellow_top": "a yellow top",
    "yellow_tshirt": "a yellow T-shirt",
    "yellow_hoodie": "a yellow hoodie",
    "yellow_sweater": "a yellow sweater",

    "black_top": "a black top",
    "black_tshirt": "a black T-shirt",
    "black_hoodie": "a black hoodie",

    "white_top": "a white top",
    "white_tshirt": "a white T-shirt",
    "white_hoodie": "a white hoodie",

    "bottom": "a clothing bottom",
    "pants": "pants",
    "blue_jeans": "blue jeans",
    "black_pants": "black pants",
    "yellow_pants": "yellow pants",

    "shorts": "shorts",
    "blue_denim_shorts": "blue denim shorts",
    "black_shorts": "black shorts",
    "yellow_shorts": "yellow shorts",

    "accessory": "an accessory",
    "bag": "a bag",
    "backpack": "a backpack",
    "red_backpack": "a red backpack",
    "black_backpack": "a black backpack",
    "yellow_backpack": "a yellow backpack",

    "handbag": "a handbag",
    "red_handbag": "a red handbag",
    "black_handbag": "a black handbag",
    "yellow_handbag": "a yellow handbag",
}


# ============================================================
# Build parent, child and leaf mappings
# ============================================================

ROOT = next(iter(HIERARCHY))

PARENT = {ROOT: None}
CHILDREN = {}


def traverse(node, subtree):
    CHILDREN[node] = list(subtree.keys())

    for child, child_subtree in subtree.items():
        PARENT[child] = node
        traverse(child, child_subtree)


traverse(ROOT, HIERARCHY[ROOT])

LEAF_IDS = [
    node
    for node, children in CHILDREN.items()
    if len(children) == 0
]

LEAF_TO_INDEX = {
    leaf_id: index
    for index, leaf_id in enumerate(LEAF_IDS)
}

LEAF_PROMPTS = [
    f"This is a photo of {NODE_LABELS[leaf_id]}."
    for leaf_id in LEAF_IDS
]


@lru_cache(maxsize=None)
def descendant_leaves(node):
    """Return every leaf beneath a node."""

    if not CHILDREN[node]:
        return (node,)

    leaves = []

    for child in CHILDREN[node]:
        leaves.extend(descendant_leaves(child))

    return tuple(leaves)


# Validate the hierarchy.
missing_labels = set(CHILDREN) - set(NODE_LABELS)
extra_labels = set(NODE_LABELS) - set(CHILDREN)

if missing_labels:
    raise ValueError(f"Missing display labels: {sorted(missing_labels)}")

if extra_labels:
    raise ValueError(f"Labels not present in hierarchy: {sorted(extra_labels)}")


# ============================================================
# Model-output compatibility
# ============================================================

def extract_embeddings(output):
    if torch.is_tensor(output):
        return output

    for attribute in (
        "text_embeds",
        "image_embeds",
        "pooler_output",
    ):
        value = getattr(output, attribute, None)

        if value is not None:
            return value

    if isinstance(output, (tuple, list)):
        return output[0]

    raise TypeError(
        f"Cannot extract embeddings from {type(output)}"
    )


# ============================================================
# HSC implementation
# ============================================================

def calculate_node_probabilities(leaf_probabilities):
    """
    Internal-node probability is the sum of all descendant
    leaf probabilities.
    """

    node_probabilities = {}

    for node in CHILDREN:
        leaf_indices = [
            LEAF_TO_INDEX[leaf]
            for leaf in descendant_leaves(node)
        ]

        node_probabilities[node] = float(
            leaf_probabilities[leaf_indices].sum()
        )

    return node_probabilities


def hsc_climbing(leaf_probabilities, threshold):
    """
    Algorithm 1 from the HSC paper:

    1. Start at the most probable leaf.
    2. While its probability is below the threshold,
       move to its parent.
    3. Return the first node satisfying the threshold.
    """

    node_probabilities = calculate_node_probabilities(
        leaf_probabilities
    )

    best_leaf_index = int(
        torch.argmax(leaf_probabilities)
    )

    best_leaf = LEAF_IDS[best_leaf_index]
    current_node = best_leaf
    climbing_path = [current_node]

    while (
        node_probabilities[current_node] < threshold
        and PARENT[current_node] is not None
    ):
        current_node = PARENT[current_node]
        climbing_path.append(current_node)

    return {
        "predicted_node": current_node,
        "predicted_label": NODE_LABELS[current_node],
        "confidence": node_probabilities[current_node],
        "best_leaf": best_leaf,
        "best_leaf_label": NODE_LABELS[best_leaf],
        "best_leaf_probability": float(
            leaf_probabilities[best_leaf_index]
        ),
        "climbing_path": climbing_path,
        "node_probabilities": node_probabilities,
    }


# ============================================================
# Test one image
# ============================================================

@torch.inference_mode()
def test_one_image(
    image_path,
    model_id,
    processor_id,
    model_loader,
    temperature,
    local=False,
):
    dtype = (
        torch.float16
        if DEVICE == "cuda"
        else torch.float32
    )
    processor = AutoProcessor.from_pretrained(processor_id)

    model = model_loader.from_pretrained(
        model_id,
        torch_dtype=dtype,
        local_files_only=local,
    ).to(DEVICE)

    model.eval()

    with Image.open(image_path) as image:
        image = ImageOps.exif_transpose(image)
        image = image.convert("RGB")

        image_inputs = processor(
            images=image,
            return_tensors="pt",
        ).to(DEVICE)

    image_embedding = extract_embeddings(
        model.get_image_features(**image_inputs)
    )

    image_embedding = F.normalize(
        image_embedding.float(),
        p=2,
        dim=-1,
    )

    # Tokenize text directly instead of through the combined processor.
    tokenizer = processor.tokenizer

    max_text_length = getattr(
        model.config.text_config,
        "max_position_embeddings",
        64,
    )

    text_inputs = tokenizer(
        LEAF_PROMPTS,
        padding="max_length",
        truncation=True,
        max_length=max_text_length,
        return_tensors="pt",
    ).to(DEVICE)

    text_embeddings = extract_embeddings(
        model.get_text_features(**text_inputs)
    )

    text_embeddings = F.normalize(
        text_embeddings.float(),
        p=2,
        dim=-1,
    )

    cosine_similarities = (
        image_embedding @ text_embeddings.T
    )[0]

    # HSC requires a probability distribution over leaves.
    leaf_probabilities = F.softmax(
        cosine_similarities / temperature,
        dim=0,
    ).cpu()

    cosine_similarities = cosine_similarities.cpu()

    top_count = min(TOP_K, len(LEAF_IDS))

    top_probabilities, top_indices = (
        leaf_probabilities.topk(top_count)
    )

    leaf_results = []

    for rank, (probability, index) in enumerate(
        zip(
            top_probabilities.tolist(),
            top_indices.tolist(),
        ),
        start=1,
    ):
        leaf_id = LEAF_IDS[index]

        leaf_results.append({
            "rank": rank,
            "leaf_id": leaf_id,
            "label": NODE_LABELS[leaf_id],
            "probability": probability,
            "cosine_similarity": float(
                cosine_similarities[index]
            ),
        })

    hsc_results = {
        threshold: hsc_climbing(
            leaf_probabilities,
            threshold,
        )
        for threshold in HSC_THRESHOLDS
    }

    del model
    del processor

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return leaf_results, hsc_results


# ============================================================
# Run all checkpoints
# ============================================================

for model_name, config in MODEL_CONFIGS.items():
    model_id = config["model_id"]
    processor_id = config["processor_id"]
    model_loader = config["loader"]
    temperature = config["temperature"]
    local = config["local"]

    print("\n" + "=" * 80)
    print(f"{model_name}: {model_id}")
    print(f"softmax temperature: {temperature}")
    print("=" * 80)

    leaf_results, hsc_results = test_one_image(
        image_path=IMAGE_PATH,
        model_id=model_id,
        processor_id=processor_id,
        model_loader=model_loader,
        temperature=temperature,
        local=local,
    )

    print("\nTop specific leaf candidates:")

    for result in leaf_results:
        print(
            f"{result['rank']:2}. "
            f"P={result['probability']:.4f} | "
            f"cos={result['cosine_similarity']:.4f} | "
            f"{result['label']}"
        )

    print("\nHSC Climbing predictions:")

    for threshold, result in hsc_results.items():
        path = " -> ".join(
            NODE_LABELS[node]
            for node in result["climbing_path"]
        )

        print(
            f"\nThreshold {threshold:.2f}: "
            f"{result['predicted_label']} "
            f"(confidence={result['confidence']:.4f})"
        )
        print(f"Path: {path}")

## 4. Fine-tuning: hierarchical multi-positive SigLIP2

Trains from `google/siglip2-base-patch16-384` (not from the prior `srpone` checkpoint) using:

- Structured label extraction from each product's caption (category, canonical colors, materials, brand,   anchored phrases) via `build_training_labels`, rather than one flat caption string.
- Product-level train/val/test split (`make_product_split`) so no product's images leak across splits.
- Identity-balanced batches: `ProductBatchSampler` draws P distinct products × K images per product per batch.
- `hierarchical_multi_positive_loss` / `directional_multi_positive_loss`: treats every valid caption/label for   the same product as a positive, instead of a standard one-image/one-caption contrastive loss.
- Configurable `TRAIN_MODE` (`heads` / `last_block` / `full`) with mode-specific learning rates, matching the   spec's staged unfreezing order.

This supersedes the earlier draft of this cell (which used flat captions with a plain multi-positive loss and no structured-label extraction) — that draft has been removed here as redundant.

In [ ]:
# Optional in a fresh Colab runtime:
# !pip install -q -U transformers accelerate safetensors sentencepiece

from pathlib import Path
from collections import defaultdict
from contextlib import nullcontext
from functools import partial
from torch.utils.data import Dataset, DataLoader, Sampler

import gc
import json
import math
import random
import re

import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image, ImageOps
from tqdm.auto import tqdm
from transformers import (
    AutoModel,
    AutoProcessor,
    get_cosine_schedule_with_warmup,
)


# ============================================================
# Configuration
# ============================================================

DATASET_ROOT = Path("/content/drive/MyDrive/apparel_dataset")
METADATA_PATH = DATASET_ROOT / "metadata.json"

BASE_MODEL_ID = "google/siglip2-base-patch16-384"

OUTPUT_DIR = DATASET_ROOT / "finetuned_siglip2_hierarchical"
BEST_MODEL_DIR = OUTPUT_DIR / "best_model"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42

# "view":
#   Every product appears in training.
#   Different images/views are held out.
#   Best for known-SKU retrieval.
#
# "product":
#   Entire products are held out.
#   Best for testing unseen-product generalization.
SPLIT_MODE = "view"

VAL_IMAGES_PER_PRODUCT = 2
TEST_IMAGES_PER_PRODUCT = 2

VAL_PRODUCT_FRACTION = 0.10
TEST_PRODUCT_FRACTION = 0.10

TRAIN_MODE = "heads"  # heads, last_block, or full
EPOCHS = 8

P_PRODUCTS = 8
K_IMAGES_PER_PRODUCT = 2

GRADIENT_ACCUMULATION_STEPS = 2
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
MAX_GRAD_NORM = 1.0

LEARNING_RATES = {
    "heads": 1e-4,
    "last_block": 1e-5,
    "full": 3e-6,
}

LEARNING_RATE = LEARNING_RATES[TRAIN_MODE]

TEXT_BATCH_SIZE = 128
IMAGE_EVAL_BATCH_SIZE = 32
SIMILARITY_BATCH_SIZE = 512

NUM_WORKERS = 2

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = DEVICE == "cuda"

# Bare product codes have no natural visual meaning.
# Keep False unless you specifically want the model to memorize
# SKU-only text queries for known products.
INCLUDE_BARE_SKU = False

LABEL_KIND_WEIGHTS = {
    "generic": 0.08,
    "attribute": 0.18,
    "brand": 0.14,
    "model": 0.20,
    "exact": 0.22,
    "clause": 0.12,
    "long": 0.06,
}

print("Device:", DEVICE)
print("Split mode:", SPLIT_MODE)
print("Training mode:", TRAIN_MODE)
print("Learning rate:", LEARNING_RATE)


# ============================================================
# Reproducibility
# ============================================================

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ============================================================
# Text parsing
#
# Structured-caption based: reads each product's `structured_caption`
# (taxonomy_path + attributes + positive_texts), generated up front by
# caption_apparel.py, instead of regex-parsing a flat caption string.
# This generalizes to all 6 brands/categories for free -- the old
# version's COLOR_ALIASES/MATERIALS/SNEAKER_TERMS/detect_category
# heuristics were shoe-only and are no longer needed.
# ============================================================


def normalize_text(text):
    text = str(text)
    text = re.sub(r"\s+", " ", text)
    return text.strip(" ,.")


def normalize_label_key(text):
    return normalize_text(text).lower()


def display_brand(brand):
    brand = normalize_text(brand)

    if not brand:
        return ""

    known = {
        "adidas": "Adidas",
        "nike": "Nike",
        "puma": "Puma",
        "reebok": "Reebok",
        "asics": "ASICS",
        "vans": "Vans",
        "converse": "Converse",
        "salomon": "Salomon",
        "saucony": "Saucony",
        "new balance": "New Balance",
        "gap": "Gap",
        "pacsun": "PacSun",
        "skechers": "Skechers",
    }

    return known.get(brand.lower(), brand.title())


def build_training_labels(product):
    structured = product.get("structured_caption") or {}

    positive_texts = [
        normalize_text(t)
        for t in structured.get("positive_texts", []) or []
        if normalize_text(t)
    ]

    taxonomy_path = [
        normalize_text(t)
        for t in structured.get("taxonomy_path", []) or []
        if normalize_text(t)
    ]

    attributes = structured.get("attributes", {}) or {}

    brand = display_brand(product.get("brand", ""))
    product_code = normalize_text(product.get("product_code", ""))
    caption = normalize_text(product.get("caption", ""))

    leaf_category = taxonomy_path[-1] if taxonomy_path else "apparel item"

    entries = []
    seen = set()

    def add(text, kind):
        text = normalize_text(text)

        if not text:
            return

        key = normalize_label_key(text)

        if key in seen:
            return

        seen.add(key)

        entries.append({
            "text": text,
            "key": key,
            "kind": kind,
        })

    # Broad hierarchy, broadest first -- e.g. footwear -> shoe -> sneaker.
    for node in taxonomy_path:
        add(node, "generic")

    # Independent attribute facets, LLM-extracted rather than regex-guessed.
    for color in attributes.get("color", []) or []:
        add(f"{color} {leaf_category}", "attribute")

    for material in attributes.get("material", []) or []:
        add(f"{material} {leaf_category}", "attribute")

    for fit in attributes.get("fit", []) or []:
        add(f"{fit} {leaf_category}", "attribute")

    for pattern in attributes.get("pattern", []) or []:
        add(f"{pattern} {leaf_category}", "attribute")

    # Brand hierarchy.
    if brand:
        add(f"{brand} {leaf_category}", "brand")

    # LLM positive_texts already span broad -> category+attrs -> brand+model
    # -> SKU-inclusive (see caption_apparel.py's prompt). Distribute them
    # across the same label_kind buckets the sampler/loss expect, by position.
    text_count = len(positive_texts)

    for position, text_item in enumerate(positive_texts):
        if text_count <= 1:
            kind = "exact"
        else:
            fraction = position / (text_count - 1)

            if fraction < 0.2:
                kind = "generic"
            elif fraction < 0.45:
                kind = "attribute"
            elif fraction < 0.7:
                kind = "brand"
            elif fraction < 0.9:
                kind = "model"
            else:
                kind = "exact"

        add(text_item, kind)

    # Identity = second-most-specific positive text (brand+model level),
    # falling back to the most specific available text.
    if len(positive_texts) >= 2:
        identity = positive_texts[-2]
    elif positive_texts:
        identity = positive_texts[-1]
    elif brand:
        identity = f"{brand} {leaf_category}"
    else:
        identity = leaf_category

    identity = normalize_text(identity)

    # Exact SKU-bearing label.
    if product_code:
        exact_label = f"{identity} ({product_code})"
        add(exact_label, "exact")

        if INCLUDE_BARE_SKU:
            add(product_code, "exact")
    elif positive_texts:
        exact_label = positive_texts[-1]
    else:
        exact_label = identity

    # Keep the original long caption too, for the "long" kind bucket.
    if caption:
        add(caption, "long")

    return {
        "entries": entries,
        "identity": identity,
        "exact_label": exact_label,
        "category": leaf_category,
        "colors": attributes.get("color", []) or [],
        "materials": attributes.get("material", []) or [],
    }


# ============================================================
# Load metadata
# ============================================================

with METADATA_PATH.open("r", encoding="utf-8") as f:
    metadata = json.load(f)


def resolve_image_path(raw_path):
    raw_path = Path(raw_path)

    candidates = [
        raw_path,
        DATASET_ROOT / raw_path,
        DATASET_ROOT.parent / raw_path,
    ]

    if (
        raw_path.parts
        and raw_path.parts[0] == DATASET_ROOT.name
    ):
        candidates.append(
            DATASET_ROOT.joinpath(*raw_path.parts[1:])
        )

    # Real files always live at DATASET_ROOT/<brand>/<slug>/<product_code>/image_N.jpg.
    # Some products still carry a stale prefix (e.g. "shoe_dataset/...") baked into
    # metadata.json from before the apparel_dataset rename -- reconstruct from the
    # last 4 path components regardless of what prefix is actually present.
    if len(raw_path.parts) >= 4:
        candidates.append(
            DATASET_ROOT.joinpath(*raw_path.parts[-4:])
        )

    for candidate in candidates:
        if candidate.is_file():
            return candidate.resolve()

    return None


records = []
product_catalog = {}
missing_paths = []

for product in metadata:
    caption = normalize_text(product.get("caption", ""))
    product_code = normalize_text(
        product.get("product_code", "")
    )

    if not product.get("structured_caption") or not product_code:
        continue

    label_data = build_training_labels(product)
    product_catalog[product_code] = label_data

    valid_label_keys = {
        entry["key"]
        for entry in label_data["entries"]
    }

    for raw_path in product.get("images", []):
        image_path = resolve_image_path(raw_path)

        if image_path is None:
            missing_paths.append(str(raw_path))
            continue

        records.append({
            "image_path": str(image_path),
            "caption": caption,
            "product_code": product_code,
            "brand": product.get("brand", ""),
            "name": product.get("name", ""),
            "training_labels": label_data["entries"],
            "valid_label_keys": valid_label_keys,
            "identity": label_data["identity"],
            "exact_label": label_data["exact_label"],
        })


if not records:
    raise RuntimeError("No valid image records were found.")


all_long_captions = list(dict.fromkeys(
    record["caption"]
    for record in records
))

all_exact_labels = list(dict.fromkeys(
    record["exact_label"]
    for record in records
))


print(f"Images: {len(records):,}")
print(f"Products: {len(product_catalog):,}")
print(f"Long captions: {len(all_long_captions):,}")
print(f"Exact labels: {len(all_exact_labels):,}")
print(f"Missing paths: {len(missing_paths):,}")


serializable_catalog = {}

for product_code, data in product_catalog.items():
    serializable_catalog[product_code] = {
        "identity": data["identity"],
        "exact_label": data["exact_label"],
        "category": data["category"],
        "colors": data["colors"],
        "materials": data["materials"],
        "labels": [
            {
                "text": entry["text"],
                "kind": entry["kind"],
            }
            for entry in data["entries"]
        ],
    }

with (
    OUTPUT_DIR / "generated_training_labels.json"
).open("w", encoding="utf-8") as f:
    json.dump(
        serializable_catalog,
        f,
        indent=2,
        ensure_ascii=False,
    )


# Show one example.
example_code = records[0]["product_code"]

print("\nExample generated labels:")
print("Product:", example_code)

for entry in product_catalog[example_code]["entries"]:
    print(f"[{entry['kind']}] {entry['text']}")


# ============================================================
# Train/validation/test split
# ============================================================

def make_view_split(image_records):
    grouped = defaultdict(list)

    for record in image_records:
        grouped[record["product_code"]].append(record)

    train = []
    validation = []
    test = []

    rng = random.Random(SEED)

    for product_code, product_records in grouped.items():
        product_records = product_records.copy()
        rng.shuffle(product_records)

        # Always preserve at least one training image.
        max_holdout = max(0, len(product_records) - 1)

        num_test = min(
            TEST_IMAGES_PER_PRODUCT,
            max_holdout,
        )

        remaining_after_test = (
            len(product_records) - num_test
        )

        num_val = min(
            VAL_IMAGES_PER_PRODUCT,
            max(0, remaining_after_test - 1),
        )

        test.extend(product_records[:num_test])

        validation.extend(
            product_records[
                num_test:num_test + num_val
            ]
        )

        train.extend(
            product_records[
                num_test + num_val:
            ]
        )

    return train, validation, test


def make_product_split(image_records):
    product_codes = sorted({
        record["product_code"]
        for record in image_records
    })

    rng = random.Random(SEED)
    rng.shuffle(product_codes)

    num_test = max(
        1,
        round(
            len(product_codes)
            * TEST_PRODUCT_FRACTION
        ),
    )

    num_val = max(
        1,
        round(
            len(product_codes)
            * VAL_PRODUCT_FRACTION
        ),
    )

    test_codes = set(product_codes[:num_test])

    val_codes = set(
        product_codes[
            num_test:num_test + num_val
        ]
    )

    train_codes = set(
        product_codes[num_test + num_val:]
    )

    train = [
        record
        for record in image_records
        if record["product_code"] in train_codes
    ]

    validation = [
        record
        for record in image_records
        if record["product_code"] in val_codes
    ]

    test = [
        record
        for record in image_records
        if record["product_code"] in test_codes
    ]

    return train, validation, test


if SPLIT_MODE == "view":
    train_records, val_records, test_records = (
        make_view_split(records)
    )
elif SPLIT_MODE == "product":
    train_records, val_records, test_records = (
        make_product_split(records)
    )
else:
    raise ValueError(
        "SPLIT_MODE must be 'view' or 'product'."
    )


print("\nSplit sizes:")
print(f"Train images: {len(train_records):,}")
print(f"Validation images: {len(val_records):,}")
print(f"Test images: {len(test_records):,}")


# ============================================================
# Dataset and sampler
# ============================================================

class ShoeDataset(Dataset):
    def __init__(self, image_records):
        self.records = image_records

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        record = self.records[index]

        with Image.open(record["image_path"]) as image:
            image = ImageOps.exif_transpose(image)
            image = image.convert("RGB")
            image = image.copy()

        return {
            "image": image,
            "record": record,
        }


class ProductBatchSampler(Sampler):
    def __init__(
        self,
        dataset,
        products_per_batch,
        images_per_product,
        seed,
    ):
        self.dataset = dataset
        self.products_per_batch = products_per_batch
        self.images_per_product = images_per_product
        self.seed = seed
        self.epoch = 0

        self.indices_by_product = defaultdict(list)

        for index, record in enumerate(dataset.records):
            self.indices_by_product[
                record["product_code"]
            ].append(index)

        self.product_codes = sorted(
            self.indices_by_product
        )

    def set_epoch(self, epoch):
        self.epoch = epoch

    def __len__(self):
        return math.ceil(
            len(self.product_codes)
            / self.products_per_batch
        )

    def __iter__(self):
        rng = random.Random(self.seed + self.epoch)

        product_codes = self.product_codes.copy()
        rng.shuffle(product_codes)

        for start in range(
            0,
            len(product_codes),
            self.products_per_batch,
        ):
            selected = product_codes[
                start:start + self.products_per_batch
            ]

            if len(selected) < self.products_per_batch:
                needed = (
                    self.products_per_batch
                    - len(selected)
                )

                selected += rng.choices(
                    product_codes,
                    k=needed,
                )

            batch_indices = []

            for product_code in selected:
                candidates = self.indices_by_product[
                    product_code
                ]

                if len(candidates) >= self.images_per_product:
                    chosen = rng.sample(
                        candidates,
                        self.images_per_product,
                    )
                else:
                    chosen = rng.choices(
                        candidates,
                        k=self.images_per_product,
                    )

                batch_indices.extend(chosen)

            rng.shuffle(batch_indices)
            yield batch_indices


processor = AutoProcessor.from_pretrained(
    BASE_MODEL_ID
)


def sample_training_label(entries):
    buckets = defaultdict(list)

    for entry in entries:
        buckets[entry["kind"]].append(entry)

    available_kinds = list(buckets.keys())

    weights = [
        LABEL_KIND_WEIGHTS.get(kind, 0.01)
        for kind in available_kinds
    ]

    selected_kind = random.choices(
        available_kinds,
        weights=weights,
        k=1,
    )[0]

    return random.choice(
        buckets[selected_kind]
    )


def training_collator(batch):
    images = [
        item["image"]
        for item in batch
    ]

    records_in_batch = [
        item["record"]
        for item in batch
    ]

    sampled_entries = [
        sample_training_label(
            record["training_labels"]
        )
        for record in records_in_batch
    ]

    sampled_texts = [
        entry["text"]
        for entry in sampled_entries
    ]

    sampled_keys = [
        entry["key"]
        for entry in sampled_entries
    ]

    processed = processor(
        images=images,
        text=sampled_texts,
        padding="max_length",
        truncation=True,
        max_length=64,
        return_tensors="pt",
    )

    batch_size = len(batch)

    positive_mask = torch.zeros(
        batch_size,
        batch_size,
        dtype=torch.bool,
    )

    # A text is positive for every image whose generated
    # label set contains that text.
    #
    # This prevents shared labels such as "yellow sneaker"
    # from being treated as negatives across products.
    for image_index, record in enumerate(
        records_in_batch
    ):
        valid_keys = record["valid_label_keys"]

        for text_index, text_key in enumerate(
            sampled_keys
        ):
            positive_mask[
                image_index,
                text_index,
            ] = text_key in valid_keys

    processed["positive_mask"] = positive_mask

    return processed


train_dataset = ShoeDataset(train_records)

train_batch_sampler = ProductBatchSampler(
    dataset=train_dataset,
    products_per_batch=P_PRODUCTS,
    images_per_product=K_IMAGES_PER_PRODUCT,
    seed=SEED,
)

train_loader = DataLoader(
    train_dataset,
    batch_sampler=train_batch_sampler,
    collate_fn=training_collator,
    num_workers=NUM_WORKERS,
    pin_memory=DEVICE == "cuda",
)


# ============================================================
# Model setup
# ============================================================

model = AutoModel.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.float32,
).to(DEVICE)


def unfreeze_module(module):
    for parameter in module.parameters():
        parameter.requires_grad = True


def configure_trainable_parameters(model, mode):
    for parameter in model.parameters():
        parameter.requires_grad = False

    if mode in {"heads", "last_block"}:
        if hasattr(model.text_model, "head"):
            unfreeze_module(model.text_model.head)

        if hasattr(
            model.text_model,
            "final_layer_norm",
        ):
            unfreeze_module(
                model.text_model.final_layer_norm
            )

        if hasattr(model.vision_model, "head"):
            unfreeze_module(model.vision_model.head)

        if hasattr(
            model.vision_model,
            "post_layernorm",
        ):
            unfreeze_module(
                model.vision_model.post_layernorm
            )

        if hasattr(model, "logit_scale"):
            model.logit_scale.requires_grad_(True)

        if hasattr(model, "logit_bias"):
            model.logit_bias.requires_grad_(True)

    if mode == "last_block":
        unfreeze_module(
            model.text_model.encoder.layers[-1]
        )

        unfreeze_module(
            model.vision_model.encoder.layers[-1]
        )

    elif mode == "full":
        for parameter in model.parameters():
            parameter.requires_grad = True

    elif mode != "heads":
        raise ValueError(
            "TRAIN_MODE must be heads, last_block, or full."
        )


configure_trainable_parameters(
    model,
    TRAIN_MODE,
)

trainable_parameters = [
    parameter
    for parameter in model.parameters()
    if parameter.requires_grad
]

trainable_count = sum(
    parameter.numel()
    for parameter in trainable_parameters
)

total_count = sum(
    parameter.numel()
    for parameter in model.parameters()
)

if trainable_count == 0:
    raise RuntimeError(
        "No model parameters were unfrozen."
    )

print(
    f"\nTrainable parameters: "
    f"{trainable_count:,} / {total_count:,} "
    f"({100 * trainable_count / total_count:.3f}%)"
)

if TRAIN_MODE != "heads":
    model.gradient_checkpointing_enable()


# ============================================================
# Multi-positive loss
# ============================================================

def directional_multi_positive_loss(
    logits,
    positive_mask,
):
    positive_mask = positive_mask.bool()
    negative_mask = ~positive_mask

    positive_counts = (
        positive_mask.sum(dim=1)
        .clamp_min(1)
    )

    negative_counts = (
        negative_mask.sum(dim=1)
        .clamp_min(1)
    )

    positive_losses = (
        -F.logsigmoid(logits)
        * positive_mask
    ).sum(dim=1) / positive_counts

    negative_losses = (
        -F.logsigmoid(-logits)
        * negative_mask
    ).sum(dim=1) / negative_counts

    return (
        0.5
        * (
            positive_losses
            + negative_losses
        )
    ).mean()


def hierarchical_multi_positive_loss(
    logits_per_image,
    positive_mask,
):
    image_to_text = directional_multi_positive_loss(
        logits_per_image,
        positive_mask,
    )

    text_to_image = directional_multi_positive_loss(
        logits_per_image.T,
        positive_mask.T,
    )

    return 0.5 * (
        image_to_text
        + text_to_image
    )


# ============================================================
# Evaluation helpers
# ============================================================

def extract_embeddings(output):
    if torch.is_tensor(output):
        return output

    for attribute in (
        "text_embeds",
        "image_embeds",
        "pooler_output",
    ):
        value = getattr(output, attribute, None)

        if value is not None:
            return value

    if isinstance(output, (tuple, list)):
        return output[0]

    raise TypeError(
        f"Cannot extract embeddings from {type(output)}"
    )


def autocast_context():
    if USE_AMP:
        return torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
        )

    return nullcontext()


def move_to_device(inputs):
    return {
        key: value.to(
            DEVICE,
            non_blocking=True,
        )
        for key, value in inputs.items()
    }


@torch.inference_mode()
def encode_texts(model, texts):
    model.eval()
    embeddings = []

    for start in range(
        0,
        len(texts),
        TEXT_BATCH_SIZE,
    ):
        batch_texts = texts[
            start:start + TEXT_BATCH_SIZE
        ]

        inputs = processor(
            text=batch_texts,
            padding="max_length",
            truncation=True,
            max_length=64,
            return_tensors="pt",
        )

        inputs = move_to_device(inputs)

        with autocast_context():
            outputs = model.get_text_features(
                **inputs
            )

            batch_embeddings = (
                extract_embeddings(outputs)
            )

        batch_embeddings = F.normalize(
            batch_embeddings.float(),
            p=2,
            dim=-1,
        )

        embeddings.append(
            batch_embeddings.cpu()
        )

    return torch.cat(embeddings, dim=0)


@torch.inference_mode()
def encode_images(model, image_records):
    model.eval()

    embeddings = []
    valid_records = []

    for start in range(
        0,
        len(image_records),
        IMAGE_EVAL_BATCH_SIZE,
    ):
        record_batch = image_records[
            start:start + IMAGE_EVAL_BATCH_SIZE
        ]

        images = []
        kept_records = []

        for record in record_batch:
            try:
                with Image.open(
                    record["image_path"]
                ) as image:
                    image = ImageOps.exif_transpose(
                        image
                    )
                    image = image.convert("RGB")
                    images.append(image.copy())

                kept_records.append(record)

            except Exception as error:
                print(
                    "Skipped:",
                    record["image_path"],
                    error,
                )

        if not images:
            continue

        inputs = processor(
            images=images,
            return_tensors="pt",
        )

        inputs = move_to_device(inputs)

        with autocast_context():
            outputs = model.get_image_features(
                **inputs
            )

            batch_embeddings = (
                extract_embeddings(outputs)
            )

        batch_embeddings = F.normalize(
            batch_embeddings.float(),
            p=2,
            dim=-1,
        )

        embeddings.append(
            batch_embeddings.cpu()
        )

        valid_records.extend(kept_records)

    return (
        torch.cat(embeddings, dim=0),
        valid_records,
    )


@torch.inference_mode()
def evaluate_single_target_retrieval(
    model,
    evaluation_records,
    candidate_labels,
    target_key,
):
    candidate_labels = list(
        dict.fromkeys(candidate_labels)
    )

    candidate_to_index = {
        label: index
        for index, label in enumerate(
            candidate_labels
        )
    }

    text_embeddings = encode_texts(
        model,
        candidate_labels,
    )

    image_embeddings, valid_records = (
        encode_images(
            model,
            evaluation_records,
        )
    )

    target_indices = torch.tensor([
        candidate_to_index[record[target_key]]
        for record in valid_records
    ])

    ranks = []

    for start in range(
        0,
        len(valid_records),
        SIMILARITY_BATCH_SIZE,
    ):
        end = min(
            start + SIMILARITY_BATCH_SIZE,
            len(valid_records),
        )

        similarities = (
            image_embeddings[start:end]
            @ text_embeddings.T
        )

        targets = target_indices[start:end]

        target_scores = similarities[
            torch.arange(end - start),
            targets,
        ]

        batch_ranks = (
            similarities
            > target_scores[:, None]
        ).sum(dim=1) + 1

        ranks.append(batch_ranks)

    ranks = torch.cat(ranks).float()

    return {
        "num_images": len(valid_records),
        "num_candidates": len(candidate_labels),
        "recall_at_1": float(
            (ranks <= 1).float().mean()
        ),
        "recall_at_5": float(
            (ranks <= 5).float().mean()
        ),
        "recall_at_10": float(
            (ranks <= 10).float().mean()
        ),
        "mrr": float(
            (1.0 / ranks).mean()
        ),
        "median_rank": float(
            ranks.median()
        ),
        "mean_rank": float(
            ranks.mean()
        ),
    }


def print_metrics(title, metrics):
    print(f"\n{title}")
    print(
        f"R@1:  "
        f"{metrics['recall_at_1'] * 100:.2f}%"
    )
    print(
        f"R@5:  "
        f"{metrics['recall_at_5'] * 100:.2f}%"
    )
    print(
        f"R@10: "
        f"{metrics['recall_at_10'] * 100:.2f}%"
    )
    print(
        f"MRR:  "
        f"{metrics['mrr'] * 100:.2f}%"
    )
    print(
        f"Median rank: "
        f"{metrics['median_rank']:.1f}"
    )
    print(
        f"Mean rank: "
        f"{metrics['mean_rank']:.2f}"
    )


# ============================================================
# Optimizer
# ============================================================

optimizer = torch.optim.AdamW(
    trainable_parameters,
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

updates_per_epoch = math.ceil(
    len(train_loader)
    / GRADIENT_ACCUMULATION_STEPS
)

total_training_steps = (
    updates_per_epoch
    * EPOCHS
)

warmup_steps = int(
    total_training_steps
    * WARMUP_RATIO
)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_training_steps,
)

scaler = torch.cuda.amp.GradScaler(
    enabled=USE_AMP
)


# ============================================================
# Baseline evaluation
# ============================================================

baseline_exact = evaluate_single_target_retrieval(
    model=model,
    evaluation_records=val_records,
    candidate_labels=all_exact_labels,
    target_key="exact_label",
)

baseline_caption = evaluate_single_target_retrieval(
    model=model,
    evaluation_records=val_records,
    candidate_labels=all_long_captions,
    target_key="caption",
)

print_metrics(
    "Validation exact-product labels before training",
    baseline_exact,
)

print_metrics(
    "Validation long captions before training",
    baseline_caption,
)


# ============================================================
# Training
# ============================================================

history = []
best_score = -1.0

for epoch in range(1, EPOCHS + 1):
    train_batch_sampler.set_epoch(epoch)

    model.train()
    optimizer.zero_grad(set_to_none=True)

    running_loss = 0.0
    batches_seen = 0

    progress = tqdm(
        train_loader,
        desc=f"Epoch {epoch}/{EPOCHS}",
    )

    for batch_index, batch in enumerate(progress):
        positive_mask = batch.pop(
            "positive_mask"
        ).to(DEVICE)

        inputs = move_to_device(batch)

        with autocast_context():
            outputs = model(**inputs)

            loss = hierarchical_multi_positive_loss(
                outputs.logits_per_image,
                positive_mask,
            )

            scaled_loss = (
                loss
                / GRADIENT_ACCUMULATION_STEPS
            )

        scaler.scale(scaled_loss).backward()

        should_update = (
            (batch_index + 1)
            % GRADIENT_ACCUMULATION_STEPS
            == 0
            or batch_index + 1
            == len(train_loader)
        )

        if should_update:
            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                trainable_parameters,
                MAX_GRAD_NORM,
            )

            scaler.step(optimizer)
            scaler.update()

            optimizer.zero_grad(
                set_to_none=True
            )

            scheduler.step()

        running_loss += float(
            loss.detach()
        )

        batches_seen += 1

        progress.set_postfix({
            "loss": (
                running_loss
                / batches_seen
            ),
            "lr": scheduler.get_last_lr()[0],
        })

    average_train_loss = (
        running_loss
        / max(batches_seen, 1)
    )

    validation_exact = (
        evaluate_single_target_retrieval(
            model=model,
            evaluation_records=val_records,
            candidate_labels=all_exact_labels,
            target_key="exact_label",
        )
    )

    validation_caption = (
        evaluate_single_target_retrieval(
            model=model,
            evaluation_records=val_records,
            candidate_labels=all_long_captions,
            target_key="caption",
        )
    )

    print_metrics(
        f"Epoch {epoch}: exact-product validation",
        validation_exact,
    )

    print_metrics(
        f"Epoch {epoch}: long-caption validation",
        validation_caption,
    )

    # Exact retrieval is weighted more heavily.
    selection_score = (
        0.70
        * validation_exact["recall_at_1"]
        + 0.30
        * validation_caption["recall_at_1"]
    )

    epoch_result = {
        "epoch": epoch,
        "train_loss": average_train_loss,
        "selection_score": selection_score,
        "exact_metrics": validation_exact,
        "caption_metrics": validation_caption,
    }

    history.append(epoch_result)

    with (
        OUTPUT_DIR / "training_history.json"
    ).open("w", encoding="utf-8") as f:
        json.dump(
            history,
            f,
            indent=2,
        )

    if selection_score > best_score:
        best_score = selection_score

        BEST_MODEL_DIR.mkdir(
            parents=True,
            exist_ok=True,
        )

        model.save_pretrained(
            BEST_MODEL_DIR,
            safe_serialization=True,
        )

        processor.save_pretrained(
            BEST_MODEL_DIR
        )

        with (
            BEST_MODEL_DIR
            / "validation_metrics.json"
        ).open("w", encoding="utf-8") as f:
            json.dump(
                {
                    "selection_score":
                        selection_score,
                    "exact_metrics":
                        validation_exact,
                    "caption_metrics":
                        validation_caption,
                },
                f,
                indent=2,
            )

        print(
            "Saved new best model:",
            BEST_MODEL_DIR,
        )


# ============================================================
# Final held-out evaluation
# ============================================================

del model
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


best_model = AutoModel.from_pretrained(
    BEST_MODEL_DIR,
    torch_dtype=torch.float32,
).to(DEVICE)

best_model.eval()

test_exact = evaluate_single_target_retrieval(
    model=best_model,
    evaluation_records=test_records,
    candidate_labels=all_exact_labels,
    target_key="exact_label",
)

test_caption = evaluate_single_target_retrieval(
    model=best_model,
    evaluation_records=test_records,
    candidate_labels=all_long_captions,
    target_key="caption",
)

print_metrics(
    "Final test: exact-product labels",
    test_exact,
)

print_metrics(
    "Final test: long captions",
    test_caption,
)

with (
    OUTPUT_DIR / "test_metrics.json"
).open("w", encoding="utf-8") as f:
    json.dump(
        {
            "exact_metrics": test_exact,
            "caption_metrics": test_caption,
        },
        f,
        indent=2,
    )

print("\nBest checkpoint:")
print(BEST_MODEL_DIR)

## 5. Evaluation: fine-tuned vs. base SigLIP2

Re-encodes the held-out test split with each candidate checkpoint and scores retrieval separately by label kind (exact product, brand, model, attribute, generic, full caption) so a strength in one granularity can't hide a weakness in another, per the spec's "keep per-task scores visible" rule (§8.3).

In [ ]:
from pathlib import Path
from collections import defaultdict

import gc
import json
import random

import torch
import torch.nn.functional as F
from PIL import Image, ImageOps
from tqdm.auto import tqdm
from transformers import AutoModel, AutoProcessor


# ============================================================
# Configuration — must match training
# ============================================================

DATASET_ROOT = Path("/content/drive/MyDrive/apparel_dataset")
METADATA_PATH = DATASET_ROOT / "metadata.json"

OUTPUT_DIR = DATASET_ROOT / "finetuned_siglip2_hierarchical"
BEST_MODEL_DIR = OUTPUT_DIR / "best_model"

GENERATED_LABELS_PATH = (
    OUTPUT_DIR / "generated_training_labels.json"
)

SEED = 42

# Must match the training run.
SPLIT_MODE = "view"  # "view" or "product"

VAL_IMAGES_PER_PRODUCT = 2
TEST_IMAGES_PER_PRODUCT = 2

VAL_PRODUCT_FRACTION = 0.10
TEST_PRODUCT_FRACTION = 0.10

TEXT_BATCH_SIZE = 128
IMAGE_BATCH_SIZE = 32
SIMILARITY_BATCH_SIZE = 512

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = (
    torch.float16
    if DEVICE == "cuda"
    else torch.float32
)

print("Device:", DEVICE)
print("Checkpoint:", BEST_MODEL_DIR)


# ============================================================
# Verify required files
# ============================================================

if not BEST_MODEL_DIR.exists():
    raise FileNotFoundError(
        f"Checkpoint does not exist: {BEST_MODEL_DIR}"
    )

if not GENERATED_LABELS_PATH.exists():
    raise FileNotFoundError(
        "Generated label file does not exist:\n"
        f"{GENERATED_LABELS_PATH}"
    )


# ============================================================
# Load metadata and previously generated labels
# ============================================================

with METADATA_PATH.open("r", encoding="utf-8") as f:
    metadata = json.load(f)

with GENERATED_LABELS_PATH.open(
    "r",
    encoding="utf-8",
) as f:
    generated_catalog = json.load(f)


def normalize_text(text):
    return " ".join(str(text).split()).strip(" ,.")


def normalize_key(text):
    return normalize_text(text).lower()


def resolve_image_path(raw_path):
    raw_path = Path(raw_path)

    candidates = [
        raw_path,
        DATASET_ROOT / raw_path,
        DATASET_ROOT.parent / raw_path,
    ]

    if (
        raw_path.parts
        and raw_path.parts[0] == DATASET_ROOT.name
    ):
        candidates.append(
            DATASET_ROOT.joinpath(*raw_path.parts[1:])
        )

    # Real files always live at DATASET_ROOT/<brand>/<slug>/<product_code>/image_N.jpg.
    # Some products still carry a stale prefix (e.g. "shoe_dataset/...") baked into
    # metadata.json from before the apparel_dataset rename -- reconstruct from the
    # last 4 path components regardless of what prefix is actually present.
    if len(raw_path.parts) >= 4:
        candidates.append(
            DATASET_ROOT.joinpath(*raw_path.parts[-4:])
        )

    for candidate in candidates:
        if candidate.is_file():
            return candidate.resolve()

    return None


records = []
missing_paths = []

for product in metadata:
    product_code = normalize_text(
        product.get("product_code", "")
    )

    caption = normalize_text(
        product.get("caption", "")
    )

    if not product_code or not caption:
        continue

    if product_code not in generated_catalog:
        print(
            "Missing generated labels for product:",
            product_code,
        )
        continue

    saved_label_data = generated_catalog[product_code]

    training_labels = []

    for entry in saved_label_data.get("labels", []):
        text = normalize_text(entry["text"])

        training_labels.append({
            "text": text,
            "key": normalize_key(text),
            "kind": entry["kind"],
        })

    exact_label = normalize_text(
        saved_label_data["exact_label"]
    )

    for raw_path in product.get("images", []):
        image_path = resolve_image_path(raw_path)

        if image_path is None:
            missing_paths.append(str(raw_path))
            continue

        records.append({
            "image_path": str(image_path),
            "product_code": product_code,
            "caption": caption,
            "exact_label": exact_label,
            "training_labels": training_labels,
        })


if not records:
    raise RuntimeError("No valid image records were found.")

print(f"Loaded images: {len(records):,}")
print(
    f"Loaded products: "
    f"{len(set(r['product_code'] for r in records)):,}"
)
print(f"Missing images: {len(missing_paths):,}")


# ============================================================
# Reconstruct the same split used during training
# ============================================================

def make_view_split(image_records):
    grouped = defaultdict(list)

    for record in image_records:
        grouped[record["product_code"]].append(record)

    train = []
    validation = []
    test = []

    rng = random.Random(SEED)

    for product_code, product_records in grouped.items():
        product_records = product_records.copy()
        rng.shuffle(product_records)

        # Always preserve at least one training image.
        max_holdout = max(
            0,
            len(product_records) - 1,
        )

        num_test = min(
            TEST_IMAGES_PER_PRODUCT,
            max_holdout,
        )

        remaining_after_test = (
            len(product_records) - num_test
        )

        num_val = min(
            VAL_IMAGES_PER_PRODUCT,
            max(0, remaining_after_test - 1),
        )

        test.extend(
            product_records[:num_test]
        )

        validation.extend(
            product_records[
                num_test:num_test + num_val
            ]
        )

        train.extend(
            product_records[
                num_test + num_val:
            ]
        )

    return train, validation, test


def make_product_split(image_records):
    product_codes = sorted({
        record["product_code"]
        for record in image_records
    })

    rng = random.Random(SEED)
    rng.shuffle(product_codes)

    num_test = max(
        1,
        round(
            len(product_codes)
            * TEST_PRODUCT_FRACTION
        ),
    )

    num_val = max(
        1,
        round(
            len(product_codes)
            * VAL_PRODUCT_FRACTION
        ),
    )

    test_codes = set(
        product_codes[:num_test]
    )

    validation_codes = set(
        product_codes[
            num_test:num_test + num_val
        ]
    )

    train_codes = set(
        product_codes[num_test + num_val:]
    )

    train = [
        record
        for record in image_records
        if record["product_code"] in train_codes
    ]

    validation = [
        record
        for record in image_records
        if record["product_code"] in validation_codes
    ]

    test = [
        record
        for record in image_records
        if record["product_code"] in test_codes
    ]

    return train, validation, test


if SPLIT_MODE == "view":
    train_records, val_records, test_records = (
        make_view_split(records)
    )

elif SPLIT_MODE == "product":
    train_records, val_records, test_records = (
        make_product_split(records)
    )

else:
    raise ValueError(
        "SPLIT_MODE must be 'view' or 'product'."
    )


print("\nReconstructed split:")
print(f"Train images: {len(train_records):,}")
print(f"Validation images: {len(val_records):,}")
print(f"Test images: {len(test_records):,}")


# ============================================================
# Load processor and checkpoint
# ============================================================

processor = AutoProcessor.from_pretrained(
    BEST_MODEL_DIR,
    local_files_only=True,
)

model = AutoModel.from_pretrained(
    BEST_MODEL_DIR,
    torch_dtype=DTYPE,
    local_files_only=True,
).to(DEVICE)

model.eval()

MAX_TEXT_LENGTH = getattr(
    model.config.text_config,
    "max_position_embeddings",
    64,
)

print("\nModel loaded.")
print("Maximum text length:", MAX_TEXT_LENGTH)


# ============================================================
# Embedding helpers
# ============================================================

def extract_embeddings(output):
    if torch.is_tensor(output):
        return output

    for attribute in (
        "text_embeds",
        "image_embeds",
        "pooler_output",
    ):
        value = getattr(output, attribute, None)

        if value is not None:
            return value

    if isinstance(output, (tuple, list)):
        return output[0]

    raise TypeError(
        f"Cannot extract embeddings from {type(output)}"
    )


@torch.inference_mode()
def encode_texts(texts):
    embeddings = []

    for start in tqdm(
        range(0, len(texts), TEXT_BATCH_SIZE),
        desc="Encoding labels",
        leave=False,
    ):
        batch_texts = texts[
            start:start + TEXT_BATCH_SIZE
        ]

        # Direct tokenizer usage avoids processor padding issues.
        inputs = processor.tokenizer(
            batch_texts,
            padding="max_length",
            truncation=True,
            max_length=MAX_TEXT_LENGTH,
            return_tensors="pt",
        ).to(DEVICE)

        output = model.get_text_features(**inputs)

        batch_embeddings = extract_embeddings(
            output
        )

        batch_embeddings = F.normalize(
            batch_embeddings.float(),
            p=2,
            dim=-1,
        )

        embeddings.append(
            batch_embeddings.cpu()
        )

    return torch.cat(embeddings, dim=0)


@torch.inference_mode()
def encode_images(image_records):
    embeddings = []
    valid_records = []
    failed_images = []

    for start in tqdm(
        range(
            0,
            len(image_records),
            IMAGE_BATCH_SIZE,
        ),
        desc="Encoding images",
        leave=False,
    ):
        batch_records = image_records[
            start:start + IMAGE_BATCH_SIZE
        ]

        images = []
        kept_records = []

        for record in batch_records:
            try:
                with Image.open(
                    record["image_path"]
                ) as image:
                    image = ImageOps.exif_transpose(
                        image
                    )

                    image = image.convert("RGB")
                    images.append(image.copy())

                kept_records.append(record)

            except Exception as error:
                failed_images.append({
                    "image_path": record["image_path"],
                    "error": repr(error),
                })

        if not images:
            continue

        inputs = processor(
            images=images,
            return_tensors="pt",
        ).to(DEVICE)

        output = model.get_image_features(**inputs)

        batch_embeddings = extract_embeddings(
            output
        )

        batch_embeddings = F.normalize(
            batch_embeddings.float(),
            p=2,
            dim=-1,
        )

        embeddings.append(
            batch_embeddings.cpu()
        )

        valid_records.extend(kept_records)

    if not embeddings:
        raise RuntimeError(
            "No images could be encoded."
        )

    return (
        torch.cat(embeddings, dim=0),
        valid_records,
        failed_images,
    )


# Encode test images once and reuse them for every label level.
(
    test_image_embeddings,
    encoded_test_records,
    failed_images,
) = encode_images(test_records)

print(
    f"Encoded test images: "
    f"{len(encoded_test_records):,}"
)

print(
    f"Failed test images: "
    f"{len(failed_images):,}"
)


# ============================================================
# Metric helpers
# ============================================================

def metrics_from_ranks(
    ranks,
    num_candidates,
):
    ranks = torch.tensor(
        ranks,
        dtype=torch.float32,
    )

    return {
        "num_images": int(len(ranks)),
        "num_candidate_labels": int(
            num_candidates
        ),
        "recall_at_1": float(
            (ranks <= 1).float().mean()
        ),
        "recall_at_5": float(
            (ranks <= 5).float().mean()
        ),
        "recall_at_10": float(
            (ranks <= 10).float().mean()
        ),
        "mrr": float(
            (1.0 / ranks).mean()
        ),
        "median_rank": float(
            ranks.median()
        ),
        "mean_rank": float(
            ranks.mean()
        ),
    }


def print_metrics(title, metrics):
    print(f"\n{title}")
    print(
        f"Images:     "
        f"{metrics['num_images']:,}"
    )
    print(
        f"Candidates: "
        f"{metrics['num_candidate_labels']:,}"
    )
    print(
        f"R@1:        "
        f"{metrics['recall_at_1'] * 100:.2f}%"
    )
    print(
        f"R@5:        "
        f"{metrics['recall_at_5'] * 100:.2f}%"
    )
    print(
        f"R@10:       "
        f"{metrics['recall_at_10'] * 100:.2f}%"
    )
    print(
        f"MRR:        "
        f"{metrics['mrr'] * 100:.2f}%"
    )
    print(
        f"Median rank: "
        f"{metrics['median_rank']:.1f}"
    )
    print(
        f"Mean rank:   "
        f"{metrics['mean_rank']:.2f}"
    )


# ============================================================
# Single-target evaluation
# ============================================================

@torch.inference_mode()
def evaluate_single_target(
    candidate_labels,
    target_key,
):
    candidate_labels = list(
        dict.fromkeys(candidate_labels)
    )

    label_to_index = {
        label: index
        for index, label in enumerate(
            candidate_labels
        )
    }

    text_embeddings = encode_texts(
        candidate_labels
    )

    ranks = []

    for start in range(
        0,
        len(encoded_test_records),
        SIMILARITY_BATCH_SIZE,
    ):
        end = min(
            start + SIMILARITY_BATCH_SIZE,
            len(encoded_test_records),
        )

        similarities = (
            test_image_embeddings[start:end]
            @ text_embeddings.T
        )

        target_indices = torch.tensor([
            label_to_index[
                encoded_test_records[index][target_key]
            ]
            for index in range(start, end)
        ])

        target_scores = similarities[
            torch.arange(end - start),
            target_indices,
        ]

        batch_ranks = (
            similarities
            > target_scores[:, None]
        ).sum(dim=1) + 1

        ranks.extend(
            batch_ranks.tolist()
        )

    return metrics_from_ranks(
        ranks,
        len(candidate_labels),
    )


# ============================================================
# Multi-target generated-label evaluation
# ============================================================

@torch.inference_mode()
def evaluate_label_kind(label_kind):
    candidate_by_key = {}

    # Candidate labels come from the complete catalog.
    for record in records:
        for entry in record["training_labels"]:
            if entry["kind"] != label_kind:
                continue

            candidate_by_key.setdefault(
                entry["key"],
                entry["text"],
            )

    candidate_keys = list(
        candidate_by_key.keys()
    )

    candidate_texts = [
        candidate_by_key[key]
        for key in candidate_keys
    ]

    if not candidate_texts:
        return None

    key_to_index = {
        key: index
        for index, key in enumerate(
            candidate_keys
        )
    }

    text_embeddings = encode_texts(
        candidate_texts
    )

    ranks = []

    for image_index, record in enumerate(
        encoded_test_records
    ):
        correct_keys = {
            entry["key"]
            for entry in record["training_labels"]
            if (
                entry["kind"] == label_kind
                and entry["key"] in key_to_index
            )
        }

        if not correct_keys:
            continue

        correct_indices = torch.tensor([
            key_to_index[key]
            for key in correct_keys
        ])

        similarities = (
            test_image_embeddings[image_index]
            @ text_embeddings.T
        )

        # An image may have several valid labels at this level.
        best_correct_score = similarities[
            correct_indices
        ].max()

        rank = int(
            (
                similarities > best_correct_score
            ).sum()
        ) + 1

        ranks.append(rank)

    if not ranks:
        return None

    return metrics_from_ranks(
        ranks,
        len(candidate_texts),
    )


# ============================================================
# Run checkpoint evaluation
# ============================================================

all_exact_labels = list(dict.fromkeys(
    record["exact_label"]
    for record in records
))

all_long_captions = list(dict.fromkeys(
    record["caption"]
    for record in records
))


test_exact = evaluate_single_target(
    candidate_labels=all_exact_labels,
    target_key="exact_label",
)

test_caption = evaluate_single_target(
    candidate_labels=all_long_captions,
    target_key="caption",
)

print_metrics(
    "Final test: exact-product labels",
    test_exact,
)

print_metrics(
    "Final test: original long captions",
    test_caption,
)


SHORT_LABEL_KINDS = [
    "generic",
    "attribute",
    "brand",
    "model",
    "clause",
]

generated_metrics = {}

for label_kind in SHORT_LABEL_KINDS:
    metrics = evaluate_label_kind(
        label_kind
    )

    if metrics is None:
        print(
            f"\nNo test labels found for kind: "
            f"{label_kind}"
        )
        continue

    generated_metrics[label_kind] = metrics

    print_metrics(
        f"Final test: {label_kind} labels",
        metrics,
    )


# ============================================================
# Save evaluation results
# ============================================================

evaluation_results = {
    "checkpoint": str(BEST_MODEL_DIR),
    "split_mode": SPLIT_MODE,
    "seed": SEED,
    "exact_metrics": test_exact,
    "caption_metrics": test_caption,
    "generated_label_metrics": generated_metrics,
    "failed_images": failed_images,
}

RESULT_PATH = (
    OUTPUT_DIR
    / "checkpoint_evaluation_all_label_levels.json"
)

with RESULT_PATH.open(
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        evaluation_results,
        f,
        indent=2,
    )


print("\nEvaluation complete.")
print("Results saved to:")
print(RESULT_PATH)


del model
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
# ============================================================
# Compare fine-tuned and original SigLIP2
# ============================================================

MODEL_CONFIGS = {
    "finetuned_siglip2": {
        "model_id": str(BEST_MODEL_DIR),
        "local_files_only": True,
    },
    "base_siglip2": {
        "model_id": "google/siglip2-base-patch16-384",
        "local_files_only": False,
    },
}

all_model_results = {}

all_exact_labels = list(dict.fromkeys(
    record["exact_label"]
    for record in records
))

all_long_captions = list(dict.fromkeys(
    record["caption"]
    for record in records
))

SHORT_LABEL_KINDS = [
    "generic",
    "attribute",
    "brand",
    "model",
    "clause",
]


for model_name, config in MODEL_CONFIGS.items():
    print("\n" + "=" * 80)
    print(model_name)
    print(config["model_id"])
    print("=" * 80)

    # These names are global because the existing helper functions
    # use model, processor, MAX_TEXT_LENGTH, test_image_embeddings,
    # and encoded_test_records directly.
    processor = AutoProcessor.from_pretrained(
        config["model_id"],
        local_files_only=config["local_files_only"],
    )

    model = AutoModel.from_pretrained(
        config["model_id"],
        torch_dtype=DTYPE,
        local_files_only=config["local_files_only"],
    ).to(DEVICE)

    model.eval()

    MAX_TEXT_LENGTH = getattr(
        model.config.text_config,
        "max_position_embeddings",
        64,
    )

    # Encode the same held-out test images for this model.
    (
        test_image_embeddings,
        encoded_test_records,
        failed_images,
    ) = encode_images(test_records)

    # Exact product.
    test_exact = evaluate_single_target(
        candidate_labels=all_exact_labels,
        target_key="exact_label",
    )

    # Original long caption.
    test_caption = evaluate_single_target(
        candidate_labels=all_long_captions,
        target_key="caption",
    )

    print_metrics(
        f"{model_name}: exact-product labels",
        test_exact,
    )

    print_metrics(
        f"{model_name}: original long captions",
        test_caption,
    )

    # Shorter generated labels.
    generated_metrics = {}

    for label_kind in SHORT_LABEL_KINDS:
        metrics = evaluate_label_kind(label_kind)

        if metrics is None:
            continue

        generated_metrics[label_kind] = metrics

        print_metrics(
            f"{model_name}: {label_kind} labels",
            metrics,
        )

    all_model_results[model_name] = {
        "model_id": config["model_id"],
        "exact_metrics": test_exact,
        "caption_metrics": test_caption,
        "generated_label_metrics": generated_metrics,
        "failed_images": failed_images,
    }

    del model
    del processor
    del test_image_embeddings

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ============================================================
# Save comparison
# ============================================================

COMPARISON_PATH = (
    OUTPUT_DIR
    / "finetuned_vs_base_siglip2.json"
)

with COMPARISON_PATH.open(
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        all_model_results,
        f,
        indent=2,
    )

print("\nComparison saved to:")
print(COMPARISON_PATH)